# Harmonizing Multi-Source Healthcare Data

Time estimate: **20** minutes

## Objectives
After completing this lab, you will be able to:
- Align healthcare datasets from multiple source systems.  
- Clean and standardize patient and encounter identifiers.  
- Map inconsistent terminologies across datasets.  
- Merge EHR, claims, and lab data into a unified view.  
- Resolve unit mismatches during data integration.


## What you will do in this lab
In this lab, you'll clean, standardize, and merge synthetic data from EHRs, insurance claims, and laboratory systems into a harmonized dataset.

You will:

- Load synthetic EHR, claims, and lab datasets.  
- Inspect and clean identifiers used across systems.  
- Standardize codes and terminology differences.  
- Resolve unit inconsistencies in lab measurements.  
- Merge datasets into a harmonized healthcare table.


## Overview
Healthcare data rarely comes from a single system. You will often need to combine information from
EHR systems, insurance claims platforms, and laboratory systems. Each source may use different
identifiers, coding standards, and measurement units. This lab focuses on the practical challenges
of harmonizing these datasets so they can be analyzed together reliably.


## About the dataset/environment
You will work with **three synthetic datasets** representing common healthcare data sources:
- EHR data with patient demographics and encounters  
- Claims data with billing and diagnosis codes  
- Lab data with test results and measurement units  

The datasets intentionally include mismatched identifiers, inconsistent codes, and unit differences.


## Setup

In [2]:
# This cell loads multi-source healthcare datasets from CSV files.
# It ensures the lab is file-driven, reproducible, and realistic.

import pandas as pd

# -----------------
# Load EHR dataset
# -----------------
ehr_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab3/ehr_data.csv")

# --------------------
# Load Claims dataset
# --------------------
claims_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab3/claims_data.csv")

# -----------------
# Load Labs dataset
# -----------------
labs_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab3/labs_data.csv")




In [3]:
# Display ehr dataset
ehr_df.head()

,ehr_patient_id,gender,age
0,P001,M,70
1,P002,F,41
2,P003,M,43
3,P004,M,77
4,P005,M,58


In [4]:
# Display claims dataset
claims_df.head()

,claims_member_id,diagnosis_code,claim_amount
0,1,I10,23589
1,2,E11,48484
2,3,I10,22453
3,4,I10,41212
4,5,E11,48525


In [5]:
# Display lab dataset
labs_df.head()

,lab_patient_id,test_name,result_value,unit
0,P-001,Glucose,5.1,mmol/L
1,P-002,Glucose,96.0,mmol/L
2,P-003,Glucose,74.0,mmol/L
3,P-004,Glucose,104.0,mmol/L
4,P-005,Glucose,101.0,mmol/L


## Step 1: Inspect identifier differences across sources
In this step, you will compare how patients are identified across EHR, claims, and lab systems.
This helps understand why records do not automatically align.

**Why this matters in healthcare:** Inconsistent identifiers are one of the most common causes of failed data integration.


In [6]:

# This cell displays patient identifiers from each dataset.
# Reviewing identifiers helps identify formatting differences early.

print(ehr_df["ehr_patient_id"], claims_df["claims_member_id"], labs_df["lab_patient_id"])


0      P001
1      P002
2      P003
3      P004
4      P005
       ... 
115    P116
116    P117
117    P118
118    P119
119    P120
Name: ehr_patient_id, Length: 120, dtype: str 0        1
1        2
2        3
3        4
4        5
      ... 
115    116
116    117
117    118
118    119
119    120
Name: claims_member_id, Length: 120, dtype: int64 0      P-001
1      P-002
2      P-003
3      P-004
4      P-005
       ...  
115    P-116
116    P-117
117    P-118
118    P-119
119    P-120
Name: lab_patient_id, Length: 120, dtype: str


## Step 2: Clean and standardize patient identifiers
Here, you will clean and standardize patient identifiers so they follow a common format.
This is a prerequisite for merging datasets reliably.

**Why this matters in healthcare:** Poor identifier hygiene can result in missing or incorrect patient matches.


In [7]:
# This cell standardizes patient identifiers across datasets.
# Consistent identifiers are required for accurate joins.

# Standardize EHR patient IDs
ehr_df["patient_id"] = ehr_df["ehr_patient_id"].str.replace("P", "")

# Standardize Lab patient IDs
labs_df["patient_id"] = labs_df["lab_patient_id"].str.replace("P-", "")

# Standardize Claims patient IDs by converting to string and padding with leading zeros
claims_df["patient_id"] = claims_df["claims_member_id"].astype(str).str.zfill(3)

ehr_df, claims_df, labs_df

(    ehr_patient_id gender  age patient_id
 0             P001      M   70        001
 1             P002      F   41        002
 2             P003      M   43        003
 3             P004      M   77        004
 4             P005      M   58        005
 ..             ...    ...  ...        ...
 115           P116      F   69        116
 116           P117      M   50        117
 117           P118      F   57        118
 118           P119      F   56        119
 119           P120      M   18        120
 
 [120 rows x 4 columns],
      claims_member_id diagnosis_code  claim_amount patient_id
 0                   1            I10         23589        001
 1                   2            E11         48484        002
 2                   3            I10         22453        003
 3                   4            I10         41212        004
 4                   5            E11         48525        005
 ..                ...            ...           ...        ...
 115            

## Step 3: Map inconsistent terminologies
Different systems might use different codes or labels for the same concept.
In this step, you will map diagnosis codes to human-readable descriptions.

**Why this matters in healthcare:** Unmapped codes make cross-system analysis difficult and error-prone.


In [8]:

# This cell maps diagnosis codes to readable descriptions.
# Terminology mapping improves interpretability.

diagnosis_map = {
    "I10": "Hypertension",
    "E11": "Type 2 Diabetes"
}

claims_df["diagnosis_desc"] = claims_df["diagnosis_code"].map(diagnosis_map)

claims_df


,claims_member_id,diagnosis_code,claim_amount,patient_id,diagnosis_desc
0,1,I10,23589,001,Hypertension
1,2,E11,48484,002,Type 2 Diabetes
2,3,I10,22453,003,Hypertension
3,4,I10,41212,004,Hypertension
4,5,E11,48525,005,Type 2 Diabetes
...,...,...,...,...,...
115,116,I10,14662,116,Hypertension
116,117,E11,21964,117,Type 2 Diabetes
117,118,I10,13130,118,Hypertension
118,119,E11,24531,119,Type 2 Diabetes


## Step 4: Resolve unit mismatches in lab data
Lab results might be reported in different units across systems.
Here, you will convert all glucose values to a single standard unit.

**Why this matters in healthcare:** Unit mismatches can lead to clinically dangerous misinterpretations.


In [9]:

# This cell standardizes lab result units.
# Converting to a common unit enables safe comparison.

def convert_glucose_to_mgdl(value, unit):
    # Convert glucose values to mg/dL
    if unit == "mmol/L":
        return value * 18
    return value

labs_df["glucose_mgdl"] = labs_df.apply(
    lambda row: convert_glucose_to_mgdl(row["result_value"], row["unit"]),
    axis=1
)

labs_df


,lab_patient_id,test_name,result_value,unit,patient_id,glucose_mgdl
0,P-001,Glucose,5.1,mmol/L,001,91.8
1,P-002,Glucose,96.0,mmol/L,002,1728.0
2,P-003,Glucose,74.0,mmol/L,003,1332.0
3,P-004,Glucose,104.0,mmol/L,004,1872.0
4,P-005,Glucose,101.0,mmol/L,005,1818.0
...,...,...,...,...,...,...
115,P-116,Glucose,85.0,mmol/L,116,1530.0
116,P-117,Glucose,6.6,mg/dL,117,6.6
117,P-118,Glucose,119.0,mmol/L,118,2142.0
118,P-119,Glucose,7.0,mg/dL,119,7.0


## Step 5: Merge EHR, claims, and lab data
With identifiers cleaned and values standardized, you will now merge the datasets.
This produces a unified patient-level view.

**Why this matters in healthcare:** Integrated data enables richer analysis across clinical and financial domains.


In [10]:

# This cell merges all datasets into a single table.
# Harmonized data supports comprehensive analytics.

merged_df = ehr_df.merge(claims_df, on="patient_id", how="inner")                   .merge(labs_df, on="patient_id", how="inner")

merged_df


,ehr_patient_id,gender,age,patient_id,claims_member_id,diagnosis_code,claim_amount,diagnosis_desc,lab_patient_id,test_name,result_value,unit,glucose_mgdl
0,P001,M,70,001,1,I10,23589,Hypertension,P-001,Glucose,5.1,mmol/L,91.8
1,P002,F,41,002,2,E11,48484,Type 2 Diabetes,P-002,Glucose,96.0,mmol/L,1728.0
2,P003,M,43,003,3,I10,22453,Hypertension,P-003,Glucose,74.0,mmol/L,1332.0
3,P004,M,77,004,4,I10,41212,Hypertension,P-004,Glucose,104.0,mmol/L,1872.0
4,P005,M,58,005,5,E11,48525,Type 2 Diabetes,P-005,Glucose,101.0,mmol/L,1818.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,P116,F,69,116,116,I10,14662,Hypertension,P-116,Glucose,85.0,mmol/L,1530.0
116,P117,M,50,117,117,E11,21964,Type 2 Diabetes,P-117,Glucose,6.6,mg/dL,6.6
117,P118,F,57,118,118,I10,13130,Hypertension,P-118,Glucose,119.0,mmol/L,2142.0
118,P119,F,56,119,119,E11,24531,Type 2 Diabetes,P-119,Glucose,7.0,mg/dL,7.0


## Step 6: Review harmonized dataset
Finally, you will review the merged dataset to confirm alignment and completeness.
This step validates the success of the harmonization process.

**Why this matters in healthcare:** Final validation prevents silent integration errors from propagating.


In [11]:

# This cell reviews the structure and contents of the harmonized dataset.
# Validation ensures integration steps worked as expected.

merged_df.info()
merged_df


<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ehr_patient_id    120 non-null    str    
 1   gender            120 non-null    str    
 2   age               120 non-null    int64  
 3   patient_id        120 non-null    str    
 4   claims_member_id  120 non-null    int64  
 5   diagnosis_code    120 non-null    str    
 6   claim_amount      120 non-null    int64  
 7   diagnosis_desc    120 non-null    str    
 8   lab_patient_id    120 non-null    str    
 9   test_name         120 non-null    str    
 10  result_value      120 non-null    float64
 11  unit              120 non-null    str    
 12  glucose_mgdl      120 non-null    float64
dtypes: float64(2), int64(3), str(8)
memory usage: 12.3 KB


,ehr_patient_id,gender,age,patient_id,claims_member_id,diagnosis_code,claim_amount,diagnosis_desc,lab_patient_id,test_name,result_value,unit,glucose_mgdl
0,P001,M,70,001,1,I10,23589,Hypertension,P-001,Glucose,5.1,mmol/L,91.8
1,P002,F,41,002,2,E11,48484,Type 2 Diabetes,P-002,Glucose,96.0,mmol/L,1728.0
2,P003,M,43,003,3,I10,22453,Hypertension,P-003,Glucose,74.0,mmol/L,1332.0
3,P004,M,77,004,4,I10,41212,Hypertension,P-004,Glucose,104.0,mmol/L,1872.0
4,P005,M,58,005,5,E11,48525,Type 2 Diabetes,P-005,Glucose,101.0,mmol/L,1818.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,P116,F,69,116,116,I10,14662,Hypertension,P-116,Glucose,85.0,mmol/L,1530.0
116,P117,M,50,117,117,E11,21964,Type 2 Diabetes,P-117,Glucose,6.6,mg/dL,6.6
117,P118,F,57,118,118,I10,13130,Hypertension,P-118,Glucose,119.0,mmol/L,2142.0
118,P119,F,56,119,119,E11,24531,Type 2 Diabetes,P-119,Glucose,7.0,mg/dL,7.0


## Exercises

In [12]:
# -----------------
# Load EHR exercise dataset
# -----------------
ehr_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab3/ehr_data_exercise.csv")

# --------------------
# Load Claims exercise dataset
# --------------------
claims_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab3/claims_data_exercise.csv")

# -----------------
# Load Labs exercise dataset
# -----------------
labs_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab3/labs_data_exercise.csv")

### Exercise 1: Inspect identifier formats

In [13]:
# your code goes here

print(ehr_df[['ehr_patient_id']])
print(claims_df[['claims_member_id']])
print(labs_df[['lab_patient_id']])

    ehr_patient_id
0            PX001
1            PX002
2            PX003
3            PX004
4            PX005
..             ...
115          PX116
116          PX117
117          PX118
118          PX119
119          PX120

[120 rows x 1 columns]
     claims_member_id
0                 501
1                 502
2                 503
3                 504
4                 505
..                ...
115               616
116               617
117               618
118               619
119               620

[120 rows x 1 columns]
    lab_patient_id
0           PX-001
1           PX-002
2           PX-003
3           PX-004
4           PX-005
..             ...
115         PX-116
116         PX-117
117         PX-118
118         PX-119
119         PX-120

[120 rows x 1 columns]


### Exercise 2: Standardize patient identifiers

In [16]:
# your code goes here

# Standardize EHR patient IDs: Remove 'PX' prefix
ehr_df["patient_id"] = ehr_df["ehr_patient_id"].str.replace("PX", "")

# Standardize Lab patient IDs: Remove 'PX-' prefix
labs_df["patient_id"] = labs_df["lab_patient_id"].str.replace("PX-", "")

# Standardize Claims patient IDs: Adjust value by subtracting 500, convert to string, and zero-fill to 3 digits
claims_df["patient_id"] = (claims_df["claims_member_id"] - 500).astype(str).str.zfill(3)

print(ehr_df.head())
print(labs_df.head())
print(claims_df.head())

  ehr_patient_id gender  age patient_id
0          PX001      F   61        001
1          PX002      F   74        002
2          PX003      F   79        003
3          PX004      M   43        004
4          PX005      F   62        005
  lab_patient_id test_name  result_value    unit patient_id
0         PX-001   Glucose           4.3  mmol/L        001
1         PX-002   Glucose           5.0  mmol/L        002
2         PX-003   Glucose         118.0  mmol/L        003
3         PX-004   Glucose           6.3  mmol/L        004
4         PX-005   Glucose         134.0  mmol/L        005
   claims_member_id diagnosis_code  claim_amount patient_id
0               501            E78         46684        001
1               502            E11         47659        002
2               503            E78         37710        003
3               504            E11         15749        004
4               505            I10         32046        005


### Exercise 3: Map diagnosis codes

In [ ]:
# your code goes here
dia_map = {
    "I10": "Hypertension",
    "E11": "Type 2 Diabetes",
    "E78": "Hyperlipidemia"
}

claims_df["diagnosis_desc"] = claims_df["diagnosis_code"].map(dia_map)

### Exercise 4: Convert lab units

In [17]:
# your code goes here

def convert_glucose_to_mgdl(value, unit):
    if unit == "mmol/L":
        return value * 18
    return value

labs_df["glucose_mgdl"] = labs_df.apply(
    lambda row: convert_glucose_to_mgdl(row["result_value"], row["unit"]), axis=1
)

labs_df[['result_value', 'unit', 'glucose_mgdl']]

,result_value,unit,glucose_mgdl
0,4.3,mmol/L,77.4
1,5.0,mmol/L,90.0
2,118.0,mmol/L,2124.0
3,6.3,mmol/L,113.4
4,134.0,mmol/L,2412.0
...,...,...,...
115,135.0,mmol/L,2430.0
116,112.0,mg/dL,112.0
117,4.9,mmol/L,88.2
118,137.0,mmol/L,2466.0


### Exercise 5: Merge datasets

In [19]:
# your code goes here

merged_df = ehr_df.merge(claims_df, on="patient_id", how="inner").merge(labs_df, on="patient_id", how="inner")

merged_df

,ehr_patient_id,gender,age,patient_id,claims_member_id,diagnosis_code,claim_amount,lab_patient_id,test_name,result_value,unit,glucose_mgdl
0,PX001,F,61,001,501,E78,46684,PX-001,Glucose,4.3,mmol/L,77.4
1,PX002,F,74,002,502,E11,47659,PX-002,Glucose,5.0,mmol/L,90.0
2,PX003,F,79,003,503,E78,37710,PX-003,Glucose,118.0,mmol/L,2124.0
3,PX004,M,43,004,504,E11,15749,PX-004,Glucose,6.3,mmol/L,113.4
4,PX005,F,62,005,505,I10,32046,PX-005,Glucose,134.0,mmol/L,2412.0
...,...,...,...,...,...,...,...,...,...,...,...,...
115,PX116,M,67,116,616,E78,13573,PX-116,Glucose,135.0,mmol/L,2430.0
116,PX117,F,41,117,617,I10,38892,PX-117,Glucose,112.0,mg/dL,112.0
117,PX118,F,62,118,618,I10,28765,PX-118,Glucose,4.9,mmol/L,88.2
118,PX119,M,31,119,619,E78,13751,PX-119,Glucose,137.0,mmol/L,2466.0


### Exercise 6: Validate merged data

In [20]:
# your code goes here

merged_df.info()
merged_df

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ehr_patient_id    120 non-null    str    
 1   gender            120 non-null    str    
 2   age               120 non-null    int64  
 3   patient_id        120 non-null    str    
 4   claims_member_id  120 non-null    int64  
 5   diagnosis_code    120 non-null    str    
 6   claim_amount      120 non-null    int64  
 7   lab_patient_id    120 non-null    str    
 8   test_name         120 non-null    str    
 9   result_value      120 non-null    float64
 10  unit              120 non-null    str    
 11  glucose_mgdl      120 non-null    float64
dtypes: float64(2), int64(3), str(7)
memory usage: 11.4 KB


,ehr_patient_id,gender,age,patient_id,claims_member_id,diagnosis_code,claim_amount,lab_patient_id,test_name,result_value,unit,glucose_mgdl
0,PX001,F,61,001,501,E78,46684,PX-001,Glucose,4.3,mmol/L,77.4
1,PX002,F,74,002,502,E11,47659,PX-002,Glucose,5.0,mmol/L,90.0
2,PX003,F,79,003,503,E78,37710,PX-003,Glucose,118.0,mmol/L,2124.0
3,PX004,M,43,004,504,E11,15749,PX-004,Glucose,6.3,mmol/L,113.4
4,PX005,F,62,005,505,I10,32046,PX-005,Glucose,134.0,mmol/L,2412.0
...,...,...,...,...,...,...,...,...,...,...,...,...
115,PX116,M,67,116,616,E78,13573,PX-116,Glucose,135.0,mmol/L,2430.0
116,PX117,F,41,117,617,I10,38892,PX-117,Glucose,112.0,mg/dL,112.0
117,PX118,F,62,118,618,I10,28765,PX-118,Glucose,4.9,mmol/L,88.2
118,PX119,M,31,119,619,E78,13751,PX-119,Glucose,137.0,mmol/L,2466.0
